In [1]:
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import seaborn as sns
import pandas as pd
import numpy as np
import yfinance as yf

In [2]:
stocks_name = ['BTC-USD', 'ETH-USD', 'SPY', 'GLD', '^VIX']

In [3]:
stocks_data = yf.download(stocks_name, start='2018-01-01', end=pd.Timestamp.now().date(), interval='1wk') #interval='5d')

[*********************100%***********************]  5 of 5 completed


In [4]:
df_stocks = pd.DataFrame(stocks_data).dropna()

In [5]:
df_stocks_close = df_stocks['Close'].dropna().drop(columns=['^VIX']).copy()

Normalized Graphic

In [6]:
normalized_df = (df_stocks_close / df_stocks_close.iloc[0])

Copy the DataFrame to a new variable

In [7]:
df_stocks_close_criptos = df_stocks['Close'].copy()

Thechnical Indicators

In [8]:
###
# Calculate SMAs for BTC-USD
df_stocks_close_criptos['BTC-USD_SMA-20'] = df_stocks_close_criptos['BTC-USD'].rolling(window=20).mean()
df_stocks_close_criptos['BTC-USD_SMA-50'] = df_stocks_close_criptos['BTC-USD'].rolling(window=50).mean()
df_stocks_close_criptos['BTC-USD_SMA-200'] = df_stocks_close_criptos['BTC-USD'].rolling(window=200).mean()
###
# Calculate SMAs for ETH-USD
df_stocks_close_criptos['ETH-USD_SMA-20'] = df_stocks_close_criptos['ETH-USD'].rolling(window=20).mean()
df_stocks_close_criptos['ETH-USD_SMA-50'] = df_stocks_close_criptos['ETH-USD'].rolling(window=50).mean()
df_stocks_close_criptos['ETH-USD_SMA-200'] = df_stocks_close_criptos['ETH-USD'].rolling(window=200).mean()
###

In [9]:
###
# Make subplot
fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.2,
    row_heights=[0.7,0.7, 0.4]
)
###
# Plot Price BTC-USD
fig.add_trace(go.Candlestick(
    x=df_stocks.index,
    open=df_stocks['Open']['BTC-USD'],
    high=df_stocks['High']['BTC-USD'],
    low=df_stocks['Low']['BTC-USD'],
    close=df_stocks['Close']['BTC-USD'],
    name='BTC-USD',

), row=1, col=1)

###
# Plot BTC SMAs
fig.add_trace(go.Scatter(
    x=df_stocks_close_criptos.index,
    y=df_stocks_close_criptos['BTC-USD_SMA-20'],
    name='SMA-20',
    line=dict(color='lightgreen')
), row=1, col=1)


fig.add_trace(go.Scatter(
    x=df_stocks_close_criptos.index,
    y=df_stocks_close_criptos['BTC-USD_SMA-50'],
    name='SMA-50',
    line=dict(color='lightblue')
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_stocks_close_criptos.index,
    y=df_stocks_close_criptos['BTC-USD_SMA-200'],
    name='SMA-200',
    line=dict(color='white')
), row=1, col=1)

###
# Plot ETH-USD
fig.add_trace(go.Candlestick(
    x=df_stocks.index,
    open=df_stocks['Open']['ETH-USD'],
    high=df_stocks['High']['ETH-USD'],
    low=df_stocks['Low']['ETH-USD'],
    close=df_stocks['Close']['ETH-USD'],
    name='ETH-USD',

), row=2, col=1)

###
# Plot ETH SMAs
fig.add_trace(go.Scatter(
    x=df_stocks_close_criptos.index,
    y=df_stocks_close_criptos['ETH-USD_SMA-20'],
    name='SMA-20',
    line=dict(color='lightgreen')
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=df_stocks_close_criptos.index,
    y=df_stocks_close_criptos['ETH-USD_SMA-50'],
    name='SMA-50',
    line=dict(color='lightblue')
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=df_stocks_close_criptos.index,
    y=df_stocks_close_criptos['ETH-USD_SMA-200'],
    name='SMA-200',
    line=dict(color='white')
), row=2, col=1)

###
# Plot VIX
fig.add_trace(go.Bar(
    x=df_stocks.index,
    y=df_stocks['Close']['^VIX'],
    name='VIX',
    marker_color=df_stocks['Close']['^VIX']
), row=3, col=1)
###
# Update layout
fig.update_layout(title='BTC-USD, ETH-USD with VIX', yaxis_title='BTC-USD', yaxis2_title='ETH-USD', yaxis3_title='VIX', height=1200, width=1200, template='plotly_dark')
fig.show()
###

Returns of the investments in multiplication

In [10]:
stocks_returns_week_df = (df_stocks_close.pct_change().dropna())#Day variation
acumulated_returns = (1 + stocks_returns_week_df).cumprod()

Log Return

In [11]:
stocks_returns_log_df = np.log(df_stocks_close/ df_stocks_close.shift(1)).dropna()

Volatility Weeks Sharpe & Sortino

In [12]:
trading_weeks_sharpe = 60 #30
trading_weeks_sortino = 90
volatility_weeks_sharpe = stocks_returns_log_df.rolling(window=trading_weeks_sharpe).std()*np.sqrt(trading_weeks_sharpe)
volatility_weeks_sortino = stocks_returns_log_df.rolling(window=trading_weeks_sortino).std()*np.sqrt(trading_weeks_sortino)

Sharpe Ratio Weeks(52)

In [13]:
rf = 0.01/52
sharpe_ratio = (stocks_returns_log_df.rolling(window=trading_weeks_sharpe).mean() - rf)*trading_weeks_sharpe/volatility_weeks_sharpe

Sortino Ratio

In [14]:
sortino_vol = stocks_returns_log_df[stocks_returns_log_df<0].rolling(window=trading_weeks_sortino, center=True, min_periods=10).std()*np.sqrt(trading_weeks_sortino)
sortino_ratio = (stocks_returns_log_df.rolling(window=trading_weeks_sortino).mean() - rf)*trading_weeks_sortino/sortino_vol

Graphic

In [15]:
# %% [markdown]
# ## Data Conditioning BTC
# %%
nv_shp_btc_top = 2.0
nv_srt_btc_top = 3.5
nv_shp_btc_bottom = -1.5
nv_srt_btc_bottom = -2.0
mask_btc_sharpe_top = sharpe_ratio['BTC-USD'] > nv_shp_btc_top
mask_btc_sharpe_sortino_top = ((sortino_ratio['BTC-USD'] >= nv_srt_btc_top) & mask_btc_sharpe_top)
mask_btc_sortino_reversion_down = (sharpe_ratio['BTC-USD'] < 1.2) & (sortino_ratio['BTC-USD'] >= nv_srt_btc_top) | ((sharpe_ratio['BTC-USD'] > nv_shp_btc_top) & (sortino_ratio['BTC-USD'] >= nv_srt_btc_top))
mask_btc_sharpe_bottom = sharpe_ratio['BTC-USD'] < nv_shp_btc_bottom
mask_btc_sortino_bottom = (sortino_ratio['BTC-USD'] < nv_srt_btc_bottom) & mask_btc_sharpe_bottom
mask_btc_sharpe_sortino_bottom = ((sortino_ratio['BTC-USD'] < nv_srt_btc_bottom) & mask_btc_sharpe_bottom)
mask_btc_sortino_reversion_up = (sharpe_ratio['BTC-USD'] > 0) & (sortino_ratio['BTC-USD'] <= -1.5)

# %% [markdown]
# ## Data Conditioning SPY
# %%

nv_shp_spy_top = 2.5
nv_srt_spy_top = 3.2
nv_shp_spy_bottom = 0
nv_srt_spy_bottom = 0
mask_spy_sharpe_top = sharpe_ratio['SPY'] > nv_shp_spy_top
mask_spy_sharpe_sortino_top = ((sortino_ratio['SPY'] >= nv_srt_spy_top) & mask_spy_sharpe_top)
mask_spy_sortino_reversion_down = (sharpe_ratio['SPY'] < nv_shp_spy_top) & (sortino_ratio['SPY'] >= nv_srt_spy_top)
mask_spy_sharpe_bottom = sharpe_ratio['SPY'] < -0.6
mask_spy_sortino_bottom = (sortino_ratio['SPY'] < nv_srt_spy_bottom)
mask_spy_sharpe_sortino_bottom = ((sortino_ratio['SPY'] < 0) & mask_spy_sharpe_bottom)
mask_spy_sortino_reversion_up = (sharpe_ratio['SPY'] > 1.0) & (sortino_ratio['SPY'] <= 0)



# %% [markdown]
# ## Data Visualization
# %%

fig = make_subplots(
    rows=6,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    row_heights=[0.5, 0.25, 0.25, 0.25, 0.25, 0.25]
)

# %% [markdown]
# ## Data Plotting Normalized Prices
# %%

fig.add_trace(
    go.Scatter(
        x = normalized_df.index,
        y = normalized_df['BTC-USD'],
        line=dict(color='purple'),
        name='BTC-USD'
    ), row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x = normalized_df.index,
        y = normalized_df['SPY'],
        line=dict(color='lightgreen'),
        name='SPY'
    ), row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x = normalized_df.index,
        y = normalized_df['GLD'],
        line=dict(color='gold'),
        name='GLD'
    ), row=1, col=1
)

# %% [markdown]
# ## Data Plotting Volatility
# %%

fig.add_trace(go.Scatter(
    x=volatility_weeks_sharpe.index,
    y=volatility_weeks_sharpe['BTC-USD'],
    line=dict(color='purple'),
    name='Volatily BTC-USD',
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=volatility_weeks_sharpe.index,
    y=volatility_weeks_sharpe['SPY'],
    line=dict(color='lightgreen'),
    name='Volatily SPY',
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=volatility_weeks_sharpe.index,
    y=volatility_weeks_sharpe['GLD'],
    line=dict(color='gold'),
    name='Volatility GLD',
), row=2, col=1)

# %% [markdown]
# ## Data Plotting Sharpe and Sortino Ratios BTC
# %%

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index,
    y=sharpe_ratio['BTC-USD'],
    line=dict(color='white'),
    name='Sharpe Ratio BTC',
), row=3, col=1)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sharpe_sortino_top],
    y=sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_top],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_top])*4,
                color='purple',
                symbol='square'),
    name='Sortino BTC > 3.5 & Sharpe > 2.0'),
    row=3, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sharpe_sortino_bottom],
    y=sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_bottom],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_bottom])*6,
                color='lightblue',
                symbol='square'),
    name='Sort < -2.0 & Shar < -2.0'),
    row=3, col=1
)

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index[mask_btc_sharpe_top],
    y=sharpe_ratio['BTC-USD'][mask_btc_sharpe_top],
    mode='markers',
    marker=dict(size=sharpe_ratio['BTC-USD'][mask_btc_sharpe_top]*3,
                color='lightcoral',
                symbol='circle'),
    name='Sharpe BTC > 2'),
    row=3, col=1
)

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index[mask_btc_sharpe_bottom],
    y=sharpe_ratio['BTC-USD'][mask_btc_sharpe_bottom],
    mode='markers',
    marker=dict(size=abs(sharpe_ratio['BTC-USD'][mask_btc_sharpe_bottom])*3,
                color='lightgreen',
                symbol='circle'),
    name='Sharpe BTC < -2'),
    row=3, col=1
)

fig.add_trace(go.Bar(
    x=sortino_ratio.index,
    y=sortino_ratio['BTC-USD'],
    marker_color=sortino_ratio['BTC-USD'],
    name='Sortino Ratio BTC',
), row=3, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sortino_reversion_down],
    y=sortino_ratio['BTC-USD'][mask_btc_sortino_reversion_down],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['BTC-USD'][mask_btc_sortino_reversion_down])*2,
                color='red',
                symbol='circle'),
    name=f'Sort > 3.5 & Shar < 0'),
    row=3, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sortino_bottom],
    y=sortino_ratio['BTC-USD'][mask_btc_sortino_bottom],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['BTC-USD'][mask_btc_sortino_bottom])*3,
                color='green',
                symbol='circle'),
    name=f'Sortino BTC < -2.0'),
    row=3, col=1
)


fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sortino_reversion_up],
    y=sortino_ratio['BTC-USD'][mask_btc_sortino_reversion_up],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['BTC-USD'][mask_btc_sortino_reversion_up])*4,
                color='lightblue',
                symbol='diamond'),
    name='Sort < -1.5 & Shar > 0'),
    row=3, col=1
)

# %% [markdown]
# ## Data Plotting Sharpe and Sortino Ratios SPY
# %%

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index,
    y=sharpe_ratio['SPY'],
    line=dict(color='white'),
    name='Sharpe Ratio SPY',
), row=4, col=1)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_spy_sharpe_sortino_top],
    y=sortino_ratio['SPY'][mask_spy_sharpe_sortino_top],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['SPY'][mask_spy_sharpe_sortino_top])*3,
                color='purple',
                symbol='square'),
    name='Sortino SPY > 3.2 & Sharpe > 2.5'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_spy_sharpe_sortino_bottom],
    y=sortino_ratio['SPY'][mask_spy_sharpe_sortino_bottom],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['SPY'][mask_spy_sharpe_sortino_bottom])*35,
                color='lightyellow',
                symbol='square'),
    name='Sort < 0 & Shar < -0.2'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index[mask_spy_sharpe_top],
    y=sharpe_ratio['SPY'][mask_spy_sharpe_top],
    mode='markers',
    marker=dict(size=sharpe_ratio['SPY'][mask_spy_sharpe_top]*3,
                color='lightcoral',
                symbol='circle'),
    name='Sharpe SPY > 2'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index[mask_spy_sharpe_bottom],
    y=sharpe_ratio['SPY'][mask_spy_sharpe_bottom],
    mode='markers',
    marker=dict(size=abs(sharpe_ratio['SPY'][mask_spy_sharpe_bottom])*8,
                color='lightgreen',
                symbol='circle'),
    name='Sharpe SPY < -0.2'),
    row=4, col=1
)

fig.add_trace(go.Bar(
    x=sortino_ratio.index,
    y=sortino_ratio['SPY'],
    marker_color=sortino_ratio['SPY'],
    name='Sortino Ratio SPY',
), row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_spy_sortino_reversion_down],
    y=sortino_ratio['SPY'][mask_spy_sortino_reversion_down],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['SPY'][mask_spy_sortino_reversion_down])*2,
                color='red',
                symbol='circle'),
    name=f'Sort > 3.5 & Shar < 0'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_spy_sortino_bottom],
    y=sortino_ratio['SPY'][mask_spy_sortino_bottom],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['SPY'][mask_spy_sortino_bottom])*12,
                color='green',
                symbol='circle'),
    name=f'Sortino SPY < 0'),
    row=4, col=1
)


fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_spy_sortino_reversion_up],
    y=sortino_ratio['SPY'][mask_spy_sortino_reversion_up],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['SPY'][mask_spy_sortino_reversion_up])*20,
                color='lightblue',
                symbol='diamond'),
    name='Sort < -1.0 & Shar > 1.0'),
    row=4, col=1
)


# %% [markdown]
# ## Data Plotting Sharpe and Sortino Ratios GLD
# %%

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index,
    y=sharpe_ratio['GLD'],
    line=dict(color='white'),
    name='Sharpe Ratio GLD',
), row=5, col=1)

fig.add_trace(go.Bar(
    x=sortino_ratio.index,
    y=sortino_ratio['GLD'],
    marker_color=sortino_ratio['GLD'],
    name='Sortino Ratio GLD',
), row=5, col=1)

# %% [markdown]
# ## Data Plotting VIX
# %%

fig.add_trace(go.Bar(
    x=df_stocks.index,
    y=df_stocks['Close']['^VIX'],
    name='VIX',
    marker_color=df_stocks['Close']['^VIX']
), row=6, col=1)

# %% [markdown]
# ## Data Plotting End
# %%

fig.update_layout(
    title= f'Volatility: {trading_weeks_sortino} weeks / Sharpe Ratio Analysis: {trading_weeks_sharpe} weeks / Sortino Ratio Analysis: {trading_weeks_sortino} weeks',
    title_x=0.5,
    xaxis_rangeslider_visible=False,
    showlegend=False,
    height=1200,
    yaxis_title='Price',
    yaxis2_title='Volatility',
    yaxis3_title='S.S Ratios BTC',
    yaxis4_title='S.S Ratios SPY',
    yaxis5_title='S.S Ratios GLD',
    yaxis6_title='VIX',
    template='plotly_dark'

)

fig.show()